# Writting a Workflow Definition

In [1]:
import os
import sys
import asyncio

### Business Logic

In [2]:
class GreetSomeone:
    async def run(self, name: str) -> str:
        return f"Hello {name}!"

In [3]:
name = input("Digite o nome:")
greeter = GreetSomeone()
greeting = await greeter.run(name)
print(greeting)

Digite o nome: qwe


Hello qwe!


## Workflow Definition

In [4]:
from temporalio import workflow


@workflow.defn
class GreetSomeone:
    @workflow.run
    async def run(self, name: str) -> str:
        return f"Hello {name}!"


In [5]:
from temporalio.client import Client
from temporalio.worker import Worker


In [6]:
client = await Client.connect(os.environ["TEMPORAL_ADDRESS"], namespace="default")

In [7]:
from temporalio.worker import Worker, UnsandboxedWorkflowRunner   
# Tem um problema com o sandbox do Temporal SDK: ele
# tenta importar o workflow a partir de __main__.__file__,
# que não existe em notebooks Jupyter. A solução é usar
# o UnsandboxedWorkflowRunner.

worker = Worker(
  client,                                                                                                                                                                                                                  
  task_queue="greeting-tasks",
  workflows=[GreetSomeone],
  workflow_runner=UnsandboxedWorkflowRunner(),
)  

In [10]:
# 

# 

Tentei rodar:
```
worker = Worker(client, task_queue="greeting-tasks", workflows=[GreetSomeone])
```

dentro do jupyter-lab (no container) mas recebei esse erro:

```
---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File /usr/local/lib/python3.10/site-packages/temporalio/worker/_workflow.py:126, in _WorkflowWorker.__init__(self, bridge_worker, namespace, task_queue, workflows, workflow_task_executor, workflow_runner, unsandboxed_workflow_runner, data_converter, interceptors, workflow_failure_exception_types, debug_mode, disable_eager_activity_execution, metric_meter, on_eviction_hook, disable_safe_eviction)
    125 if defn.sandboxed:
--> 126     workflow_runner.prepare_workflow(defn)
    127 else:

File /usr/local/lib/python3.10/site-packages/temporalio/worker/workflow_sandbox/_runner.py:76, in SandboxedWorkflowRunner.prepare_workflow(self, defn)
     75 # Just create with fake info which validates
---> 76 self.create_instance(
     77     WorkflowInstanceDetails(
     78         payload_converter_class=temporalio.converter.DataConverter.default.payload_converter_class,
     79         failure_converter_class=temporalio.converter.DataConverter.default.failure_converter_class,
     80         interceptor_classes=[],
     81         defn=defn,
     82         # Just use fake info during validation
     83         info=_fake_info,
     84         randomness_seed=-1,
     85         extern_functions={},
     86         disable_eager_activity_execution=False,
     87         worker_level_failure_exception_types=self._worker_level_failure_exception_types,
     88     ),
     89 )

File /usr/local/lib/python3.10/site-packages/temporalio/worker/workflow_sandbox/_runner.py:93, in SandboxedWorkflowRunner.create_instance(self, det)
     92 """Implements :py:meth:`WorkflowRunner.create_instance`."""
---> 93 return _Instance(det, self._runner_class, self._restrictions)

File /usr/local/lib/python3.10/site-packages/temporalio/worker/workflow_sandbox/_runner.py:119, in _Instance.__init__(self, instance_details, runner_class, restrictions)
    116 self.globals_and_locals = {
    117     "__file__": "workflow_sandbox.py",
    118 }
--> 119 self._create_instance()

File /usr/local/lib/python3.10/site-packages/temporalio/worker/workflow_sandbox/_runner.py:130, in _Instance._create_instance(self)
    128 try:
    129     # Import user code
--> 130     self._run_code(
    131         "with __temporal_importer.applied():\n"
    132         # Import the workflow code
    133         f"  from {module_name} import {self.instance_details.defn.cls.__name__} as __temporal_workflow_class\n"
    134         f"  from {self.runner_class.__module__} import {self.runner_class.__name__} as __temporal_runner_class\n",
    135         __temporal_importer=self.importer,
    136     )
    138     # Set context as in runtime

File /usr/local/lib/python3.10/site-packages/temporalio/worker/workflow_sandbox/_runner.py:172, in _Instance._run_code(self, code, **extra_globals)
    171     temporalio.workflow.unsafe._set_in_sandbox(True)
--> 172     exec(code, self.globals_and_locals, self.globals_and_locals)
    173 finally:

File <string>:2

File /usr/local/lib/python3.10/site-packages/temporalio/worker/workflow_sandbox/_importer.py:441, in _ThreadLocalCallable.__call__(self, *args, **kwargs)
    440 def __call__(self, *args: _P.args, **kwargs: _P.kwargs) -> _T:
--> 441     return self.current(*args, **kwargs)

File /usr/local/lib/python3.10/site-packages/temporalio/worker/workflow_sandbox/_importer.py:220, in Importer._import(self, name, globals, locals, fromlist, level)
    218 orig_mod = _thread_local_sys_modules.orig["__main__"]
    219 new_spec = importlib.util.spec_from_file_location(
--> 220     full_name, orig_mod.__file__
    221 )
    222 if not new_spec:

AttributeError: module '__main__' has no attribute '__file__'

The above exception was the direct cause of the following exception:

RuntimeError                              Traceback (most recent call last)
Cell In[8], line 1
----> 1 worker = Worker(client, task_queue="greeting-tasks", workflows=[GreetSomeone])

File /usr/local/lib/python3.10/site-packages/temporalio/worker/_worker.py:298, in Worker.__init__(self, client, task_queue, activities, workflows, activity_executor, workflow_task_executor, workflow_runner, unsandboxed_workflow_runner, interceptors, build_id, identity, max_cached_workflows, max_concurrent_workflow_tasks, max_concurrent_activities, max_concurrent_local_activities, max_concurrent_workflow_task_polls, nonsticky_to_sticky_poll_ratio, max_concurrent_activity_task_polls, no_remote_activities, sticky_queue_schedule_to_start_timeout, max_heartbeat_throttle_interval, default_heartbeat_throttle_interval, max_activities_per_second, max_task_queue_activities_per_second, graceful_shutdown_timeout, workflow_failure_exception_types, shared_state_manager, debug_mode, disable_eager_activity_execution, on_fatal_error, use_worker_versioning, disable_safe_workflow_eviction)
    296 self._workflow_worker: Optional[_WorkflowWorker] = None
    297 if workflows:
--> 298     self._workflow_worker = _WorkflowWorker(
    299         bridge_worker=lambda: self._bridge_worker,
    300         namespace=client.namespace,
    301         task_queue=task_queue,
    302         workflows=workflows,
    303         workflow_task_executor=workflow_task_executor,
    304         workflow_runner=workflow_runner,
    305         unsandboxed_workflow_runner=unsandboxed_workflow_runner,
    306         data_converter=client_config["data_converter"],
    307         interceptors=interceptors,
    308         workflow_failure_exception_types=workflow_failure_exception_types,
    309         debug_mode=debug_mode,
    310         disable_eager_activity_execution=disable_eager_activity_execution,
    311         metric_meter=self._runtime.metric_meter,
    312         on_eviction_hook=None,
    313         disable_safe_eviction=disable_safe_workflow_eviction,
    314     )
    316 # Create bridge worker last. We have empirically observed that if it is
    317 # created before an error is raised from the activity worker
    318 # constructor, a deadlock/hang will occur presumably while trying to
    319 # free it.
    320 # TODO(cretz): Why does this cause a test hang when an exception is
    321 # thrown after it?
    322 assert bridge_client._bridge_client

File /usr/local/lib/python3.10/site-packages/temporalio/worker/_workflow.py:130, in _WorkflowWorker.__init__(self, bridge_worker, namespace, task_queue, workflows, workflow_task_executor, workflow_runner, unsandboxed_workflow_runner, data_converter, interceptors, workflow_failure_exception_types, debug_mode, disable_eager_activity_execution, metric_meter, on_eviction_hook, disable_safe_eviction)
    128         unsandboxed_workflow_runner.prepare_workflow(defn)
    129 except Exception as err:
--> 130     raise RuntimeError(f"Failed validating workflow {defn.name}") from err
    131 if defn.name:
    132     self._workflows[defn.name] = defn

RuntimeError: Failed validating workflow GreetSomeone
```